In [1]:
!pip install streamlit pyngrok pyjwt bcrypt watchdog

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 32.1 MB/s eta 0:00:00


In [2]:
!pip install streamlit pyngrok bcrypt pyjwt

In [3]:
%%writefile app.py
import streamlit as st
import sqlite3
import re
import jwt
import datetime
import time
import bcrypt

# ==========================================
# CONFIGURATION
# ==========================================
SECRET_KEY = "super_secret_key"
ALGORITHM = "HS256"
TOKEN_EXPIRE_MINUTES = 30

# ==========================================
# DATABASE
# ==========================================
conn = sqlite3.connect("users.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS users(
    username TEXT NOT NULL,
    email TEXT PRIMARY KEY,
    password BLOB NOT NULL,
    security_question TEXT NOT NULL,
    security_answer TEXT NOT NULL
)
""")
conn.commit()

# ==========================================
# JWT
# ==========================================
def create_token(email):
    payload = {
        "email": email,
        "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=TOKEN_EXPIRE_MINUTES)
    }
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)

def verify_token(token):
    try:
        return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
    except:
        return None

# ==========================================
# VALIDATION
# ==========================================
def valid_email(email):
    pattern = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'
    return re.fullmatch(pattern, email)

def valid_password(password):
    return password.isalnum() and len(password) >= 8

# ==========================================
# HASHING
# ==========================================
def hash_password(password):
    return bcrypt.hashpw(password.encode(), bcrypt.gensalt())

def check_password(password, hashed):
    return bcrypt.checkpw(password.encode(), hashed)

# ==========================================
# PAGE CONFIG
# ==========================================
st.set_page_config(page_title="TextMorph", layout="wide")

# ==========================================
# CSS
# ==========================================
st.markdown("""
<style>
@keyframes gradientBG {
  0% {background-position: 0% 50%;}
  50% {background-position: 100% 50%;}
  100% {background-position: 0% 50%;}
}

.stApp {
    background: linear-gradient(-45deg, #141E30, #243B55, #4F8BF9, #6A5ACD);
    background-size: 400% 400%;
    animation: gradientBG 15s ease infinite;
}

.hero {
    text-align:center;
    padding:60px 20px;
    color:white;
}

.glass {
    background: rgba(255,255,255,0.1);
    backdrop-filter: blur(18px);
    border-radius:20px;
    padding:40px;
    box-shadow:0 8px 32px rgba(0,0,0,0.4);
    max-width:500px;
    margin:auto;
}

.stButton>button {
    width:100%;
    border-radius:12px;
    height:45px;
    background:linear-gradient(90deg,#4F8BF9,#6A5ACD);
    color:white;
    font-weight:bold;
}

.error-box {
    background:#ff4b4b;
    color:white;
    padding:10px;
    border-radius:10px;
    margin-bottom:8px;
}

.success-box {
    background:#00c853;
    color:white;
    padding:10px;
    border-radius:10px;
}
</style>
""", unsafe_allow_html=True)

# ==========================================
# SESSION
# ==========================================
if "page" not in st.session_state:
    st.session_state.page = "login"
if "token" not in st.session_state:
    st.session_state.token = None
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []
if "reset_step" not in st.session_state:
    st.session_state.reset_step = 1

# ==========================================
# SIGNUP
# ==========================================
def signup():
    st.markdown('<div class="hero"><h1>Create Account</h1></div>', unsafe_allow_html=True)
    st.markdown('<div class="glass">', unsafe_allow_html=True)

    username = st.text_input("Username *")
    email = st.text_input("Email *")
    password = st.text_input("Password *", type="password")
    confirm = st.text_input("Confirm Password *", type="password")

    question = st.selectbox("Security Question *", [
        "What is your pet name?",
        "What is your mother’s maiden name?",
        "What is your favorite teacher?"
    ])
    answer = st.text_input("Security Answer *")

    if st.button("Sign Up"):
        errors=[]

        if not username: errors.append("Username mandatory.")
        if not email: errors.append("Email mandatory.")
        elif not valid_email(email): errors.append("Invalid email format.")
        if not password: errors.append("Password mandatory.")
        elif not valid_password(password): errors.append("Password must be alphanumeric & min 8.")
        if password != confirm: errors.append("Passwords must match.")
        if not answer: errors.append("Security answer mandatory.")

        cursor.execute("SELECT * FROM users WHERE email=?", (email,))
        if cursor.fetchone(): errors.append("Email already registered.")

        if errors:
            for e in errors:
                st.markdown(f'<div class="error-box">{e}</div>', unsafe_allow_html=True)
        else:
            hashed = hash_password(password)
            cursor.execute("INSERT INTO users VALUES (?,?,?,?,?)",
                           (username,email,hashed,question,answer))
            conn.commit()
            st.session_state.token=create_token(email)
            st.markdown('<div class="success-box">Signup Successful!</div>', unsafe_allow_html=True)
            time.sleep(1)
            st.rerun()

    if st.button("Back to Login"):
        st.session_state.page="login"
        st.rerun()

    st.markdown('</div>', unsafe_allow_html=True)

# ==========================================
# LOGIN
# ==========================================
def login():
    st.markdown('''<div class="hero">
        <h1>Welcome to TextMorph ⭐</h1>
        <p>Secure AI Workspace</p>
    </div>''', unsafe_allow_html=True)

    st.markdown('<div class="glass">', unsafe_allow_html=True)

    email = st.text_input("Email *")
    password = st.text_input("Password *", type="password")

    if st.button("Login"):
        if not email:
            st.markdown('<div class="error-box">Email is required.</div>', unsafe_allow_html=True)
        elif not valid_email(email):
            st.markdown('<div class="error-box">Invalid email format.</div>', unsafe_allow_html=True)
        elif not password:
            st.markdown('<div class="error-box">Password is required.</div>', unsafe_allow_html=True)
        else:
            cursor.execute("SELECT password FROM users WHERE email=?", (email,))
            user = cursor.fetchone()

            if not user:
                st.markdown('<div class="error-box">Email not found.</div>', unsafe_allow_html=True)
                if st.button("Create Account Now"):
                    st.session_state.page="signup"
                    st.rerun()

            elif not check_password(password,user[0]):
                st.markdown('<div class="error-box">Incorrect password.</div>', unsafe_allow_html=True)

            else:
                st.session_state.token=create_token(email)
                st.markdown('<div class="success-box">Login Successful!</div>', unsafe_allow_html=True)
                time.sleep(1)
                st.rerun()

    col1,col2=st.columns(2)
    with col1:
        if st.button("Forgot Password?"):
            st.session_state.page="forgot"
            st.rerun()
    with col2:
        if st.button("Create Account"):
            st.session_state.page="signup"
            st.rerun()

    st.markdown('</div>', unsafe_allow_html=True)

# ==========================================
# FORGOT PASSWORD
# ==========================================
def forgot_password():
    st.markdown('<div class="hero"><h1>Reset Password 🔐</h1></div>', unsafe_allow_html=True)
    st.markdown('<div class="glass">', unsafe_allow_html=True)

    if st.session_state.reset_step == 1:
        email = st.text_input("Enter Registered Email *")
        if st.button("Verify Email"):
            cursor.execute("SELECT security_question FROM users WHERE email=?", (email,))
            result=cursor.fetchone()
            if not result:
                st.markdown('<div class="error-box">Email not found.</div>', unsafe_allow_html=True)
            else:
                st.session_state.reset_email=email
                st.session_state.security_question=result[0]
                st.session_state.reset_step=2
                st.rerun()

    elif st.session_state.reset_step == 2:
        st.write(st.session_state.security_question)
        answer=st.text_input("Enter Security Answer *")
        if st.button("Verify Answer"):
            cursor.execute("SELECT security_answer FROM users WHERE email=?",(st.session_state.reset_email,))
            correct=cursor.fetchone()[0]
            if answer.strip().lower()!=correct.strip().lower():
                st.markdown('<div class="error-box">Incorrect answer.</div>', unsafe_allow_html=True)
            else:
                st.session_state.reset_step=3
                st.rerun()

    elif st.session_state.reset_step == 3:
        new_pass=st.text_input("New Password *",type="password")
        confirm=st.text_input("Confirm Password *",type="password")

        if st.button("Reset Password"):
            if not valid_password(new_pass):
                st.markdown('<div class="error-box">Password must be alphanumeric & min 8.</div>', unsafe_allow_html=True)
            elif new_pass!=confirm:
                st.markdown('<div class="error-box">Passwords do not match.</div>', unsafe_allow_html=True)
            else:
                hashed=hash_password(new_pass)
                cursor.execute("UPDATE users SET password=? WHERE email=?",
                               (hashed,st.session_state.reset_email))
                conn.commit()
                st.markdown('<div class="success-box">Password Reset Successful!</div>', unsafe_allow_html=True)
                st.session_state.reset_step=1
                st.session_state.page="login"
                time.sleep(1)
                st.rerun()

    if st.button("Back to Login"):
        st.session_state.reset_step=1
        st.session_state.page="login"
        st.rerun()

    st.markdown('</div>', unsafe_allow_html=True)

# ==========================================
# DASHBOARD
# ==========================================
# ==========================================
# DASHBOARD
# ==========================================
def dashboard():
    payload = verify_token(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.rerun()

    email = payload["email"]
    cursor.execute("SELECT username FROM users WHERE email=?", (email,))
    username = cursor.fetchone()[0]

    # ==============================
    # SIDEBAR (Left Corner Menu)
    # ==============================
    with st.sidebar:
        st.markdown("## 🤖 TextMorph")
        st.markdown("---")

        if st.button("💬 Chat Now"):
            st.session_state.page = "chat"

        if st.button("🆕 New Chat"):
            st.session_state.chat_history = []

        if st.button("🚪 Logout"):
            st.session_state.token = None
            st.session_state.page = "login"
            st.rerun()

    # ==============================
    # MAIN DASHBOARD AREA
    # ==============================

  # ==========================================
# DASHBOARD
# ==========================================
def dashboard():
    payload = verify_token(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.rerun()

    email = payload["email"]
    cursor.execute("SELECT username FROM users WHERE email=?", (email,))
    username = cursor.fetchone()[0]

    # ==============================
    # SIDEBAR (Left Corner Menu)
    # ==============================
    with st.sidebar:
        st.markdown("## 🤖 TextMorph")
        st.markdown("---")

        if st.button("💬 Chat Now"):
            st.session_state.page = "chat"

        if st.button("🆕 New Chat"):
            st.session_state.chat_history = []

        if st.button("🚪 Logout"):
            st.session_state.token = None
            st.session_state.page = "login"
            st.rerun()

    # ==============================
    # MAIN DASHBOARD AREA
    # ==============================

  # ==========================================
# DASHBOARD
# ==========================================
def dashboard():
    payload = verify_token(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.rerun()

    email = payload["email"]
    cursor.execute("SELECT username FROM users WHERE email=?", (email,))
    username = cursor.fetchone()[0]

    # ==============================
    # SIDEBAR (Left Corner Menu)
    # ==============================
    with st.sidebar:
        st.markdown("## 🤖 TextMorph")
        st.markdown("---")

        if st.button("💬 Chat Now"):
            st.session_state.page = "chat"

        if st.button("🆕 New Chat"):
            st.session_state.chat_history = []

        if st.button("🚪 Logout"):
            st.session_state.token = None
            st.session_state.page = "login"
            st.rerun()

    # ==============================
    # MAIN DASHBOARD AREA
    # ==============================

    st.markdown(
        f'<div class="hero"><h1>Welcome, {username} 👋</h1></div>',
        unsafe_allow_html=True
    )

    # If Chat selected OR default
    if st.session_state.page == "chat" or "chat" not in st.session_state.page:

        for role, msg in st.session_state.chat_history:
            st.chat_message(role).write(msg)

        prompt = st.chat_input("Ask TextMorph LLM...")
        if prompt:
            st.session_state.chat_history.append(("user", prompt))
            response = "This is a demo LLM response."
            st.session_state.chat_history.append(("assistant", response))
            st.rerun()
# ==========================================
# ROUTER
# ==========================================
if st.session_state.token:
    dashboard()
else:
    if st.session_state.page=="signup":
        signup()
    elif st.session_state.page=="forgot":
        forgot_password()
    else:
        login()

Writing app.py


In [4]:
!streamlit run app.py --server.port 8501 --server.headless true &>/dev/null &

In [5]:
from pyngrok import ngrok

# Kill old tunnels if any
ngrok.kill()

# Add your personal auth token here
ngrok.set_auth_token("")

# Create tunnel
public_url = ngrok.connect(8501)
print("🌍 Your App URL:", public_url)

🌍 Your App URL: NgrokTunnel: "https://contradistinctive-noninterceptive-vihaan.ngrok-free.dev" -> "http://localhost:8501"
